In [1]:
import torch
import torch.nn as nn

from src.quadratic_annealing_optimizer import QuadraticAnnealingOptimizer
from src.models import QuadraticMLP
from src.utils import build_sampler, data_load_and_prep
from src.training import train
from src.newton_optimizer import NewtonOptimizer

EXPERIMENT_NAME = "optimizer-comparison-mlp-iris"

## Idea

For the current parameter vector $w$, the batch loss is approximated locally by a second-order Taylor expansion:

$$
\mathcal{L}(w + \Delta) \approx \mathcal{L}(w) + g^\top \Delta + \frac{1}{2}\Delta^\top H\Delta,
$$

where $g = \nabla \mathcal{L}(w)$ and $H = \nabla^2 \mathcal{L}(w)$.

The annealer does not optimize the full continuous update directly. Instead, for a selected subset of parameters, it chooses binary variables $z_i \in \{0,1\}$ that encode the sign of a fixed-size step:

$$
\Delta_i = \eta(2z_i - 1),
$$

so $z_i = 1$ means a $+\eta$ step and $z_i = 0$ means a $-\eta$ step. Substituting this into the quadratic model turns the local optimization problem into a QUBO/BQM:

$$
E(z) = \sum_i a_i z_i + \sum_{i<j} b_{ij} z_i z_j + c.
$$

The annealer minimizes $E(z)$, then the proposed update is applied to the network parameters and accepted only if the true loss decreases.

## Loading sample Iris data for training

In [2]:
train_loader, test_loader = data_load_and_prep("iris", test_size=0.3, random_state=42, batch_size='full')

## Training

In [3]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [4], 3)
classical_device = "cuda" 

sampler = build_sampler(mode="simulated")

optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cpu",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [4]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="quadratic-annealer-optimizer"
)

/home/filip/studia/master/second-order-by-annealer/venv/lib/python3.12/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


Epoch 000 | train_loss=1.0255 | train_acc=0.638 | test_loss=1.0259 | test_acc=0.689 | 
Epoch 005 | train_loss=0.7107 | train_acc=0.771 | test_loss=0.7151 | test_acc=0.778 | 
Epoch 010 | train_loss=0.5188 | train_acc=0.810 | test_loss=0.5214 | test_acc=0.800 | 
Epoch 015 | train_loss=0.3941 | train_acc=0.876 | test_loss=0.3992 | test_acc=0.867 | 
Epoch 020 | train_loss=0.2996 | train_acc=0.924 | test_loss=0.3023 | test_acc=0.889 | 
Epoch 025 | train_loss=0.2182 | train_acc=0.933 | test_loss=0.1954 | test_acc=0.956 | 


2026/04/07 22:36:10 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=0.1654 | train_acc=0.952 | test_loss=0.1277 | test_acc=1.000 | 


In [6]:
loss_fn = nn.CrossEntropyLoss()
model = QuadraticMLP(4, [4], 3)
classical_device = "cuda" 

sampler = build_sampler(mode="gpu_simulated")
optimizer = QuadraticAnnealingOptimizer(
    sampler=sampler,
    model=model,
    device="cuda",
    subset_size=12,
    step_size=0.05,
    num_reads=100,
)

In [7]:
train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="gpu-quadratic-annealer-optimizer"
)

Epoch 000 | train_loss=1.0757 | train_acc=0.448 | test_loss=1.1309 | test_acc=0.289 | 
Epoch 005 | train_loss=0.8364 | train_acc=0.724 | test_loss=0.8430 | test_acc=0.711 | 
Epoch 010 | train_loss=0.6009 | train_acc=0.762 | test_loss=0.5725 | test_acc=0.733 | 
Epoch 015 | train_loss=0.4613 | train_acc=0.800 | test_loss=0.4365 | test_acc=0.800 | 
Epoch 020 | train_loss=0.3655 | train_acc=0.867 | test_loss=0.3526 | test_acc=0.844 | 
Epoch 025 | train_loss=0.2930 | train_acc=0.924 | test_loss=0.2813 | test_acc=0.889 | 


2026/04/07 22:36:33 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=0.2384 | train_acc=0.924 | test_loss=0.2262 | test_acc=0.911 | 


# Classical optimization for benchmarking

In [8]:
loss_fn = nn.CrossEntropyLoss()
adam_model = QuadraticMLP(4, [4], 3)
classical_device = "cuda" 
adam_optimizer = torch.optim.Adam(adam_model.parameters(), 
                             lr=0.01,
                             betas=[0.9, 0.999],
                             )

## Optimization using Adam optimizaer

In [9]:
train(
    model=adam_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=adam_optimizer,
    c_device=classical_device,
    epochs=30, 
    experiment_name = EXPERIMENT_NAME,
    run_name="adam-optimizer"
)

2026/04/07 22:36:37 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.2834 | train_acc=0.343 | test_loss=1.3683 | test_acc=0.289 | 
Epoch 005 | train_loss=1.1551 | train_acc=0.352 | test_loss=1.2206 | test_acc=0.289 | 
Epoch 010 | train_loss=1.0420 | train_acc=0.352 | test_loss=1.0849 | test_acc=0.289 | 
Epoch 015 | train_loss=0.9410 | train_acc=0.695 | test_loss=0.9604 | test_acc=0.689 | 
Epoch 020 | train_loss=0.8531 | train_acc=0.790 | test_loss=0.8517 | test_acc=0.778 | 
Epoch 025 | train_loss=0.7777 | train_acc=0.790 | test_loss=0.7614 | test_acc=0.800 | 
Epoch 029 | train_loss=0.7246 | train_acc=0.781 | test_loss=0.7018 | test_acc=0.778 | 


## Optimization using LBFGS-style second-order optimizer

In [10]:
loss_fn = nn.CrossEntropyLoss()
lbfgs_model = QuadraticMLP(4, [4], 3)
classical_device = "cuda" 
lbfgs_optimizer = torch.optim.LBFGS(lbfgs_model.parameters(), 
                              lr=0.01,
                              max_iter=1,
                              history_size=100,
                              line_search_fn=None,
                              )

In [11]:
train(
    model=lbfgs_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=lbfgs_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="lbfgs-optimizer"
)

2026/04/07 22:36:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 000 | train_loss=1.1021 | train_acc=0.419 | test_loss=1.0667 | test_acc=0.422 | 
Epoch 005 | train_loss=0.9450 | train_acc=0.648 | test_loss=0.8928 | test_acc=0.711 | 
Epoch 010 | train_loss=0.8568 | train_acc=0.648 | test_loss=0.7967 | test_acc=0.711 | 
Epoch 015 | train_loss=0.7850 | train_acc=0.648 | test_loss=0.7198 | test_acc=0.711 | 
Epoch 020 | train_loss=0.7314 | train_acc=0.648 | test_loss=0.6632 | test_acc=0.711 | 
Epoch 025 | train_loss=0.6901 | train_acc=0.648 | test_loss=0.6205 | test_acc=0.711 | 
Epoch 029 | train_loss=0.6599 | train_acc=0.648 | test_loss=0.5900 | test_acc=0.711 | 


## Optimization using Newton-Rapson method

In [16]:
loss_fn = nn.CrossEntropyLoss() 
newton_model = QuadraticMLP(4, [4], 3)
classical_device = "cuda"
newton_optimizer = NewtonOptimizer(newton_model.parameters(), 
                                   lr=1, 
                                   max_iter=1,
                                   damping=1e-4,
                                   )

In [15]:
train(
    model=newton_model,
    train_loader=train_loader,
    test_loader=test_loader,
    loss_fn=loss_fn,
    optimizer=newton_optimizer,
    c_device=classical_device,
    epochs=30,
    experiment_name = EXPERIMENT_NAME,
    run_name="newton-optimizer"
)

Epoch 000 | train_loss=1.1669 | train_acc=0.381 | test_loss=1.1209 | test_acc=0.444 | 
Epoch 005 | train_loss=1.0894 | train_acc=0.343 | test_loss=1.1135 | test_acc=0.244 | 
Epoch 010 | train_loss=1.0855 | train_acc=0.352 | test_loss=1.1150 | test_acc=0.289 | 
Epoch 015 | train_loss=1.0854 | train_acc=0.352 | test_loss=1.1159 | test_acc=0.289 | 
Epoch 020 | train_loss=1.0853 | train_acc=0.352 | test_loss=1.1160 | test_acc=0.289 | 
Epoch 025 | train_loss=1.0850 | train_acc=0.362 | test_loss=1.1161 | test_acc=0.289 | 


2026/04/07 22:40:31 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Epoch 029 | train_loss=1.0852 | train_acc=0.362 | test_loss=1.1161 | test_acc=0.289 | 
